# 05 - Evaluation (v3)

Thin Kaggle runner. Evaluation logic lives in `src/eval/run_evaluation.py`.


In [ ]:
!pip install -q -U datasets transformers trl peft accelerate bitsandbytes pyyaml pandas matplotlib scikit-learn rouge_score
!pip install -q -U --no-deps unsloth


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/W-Kaski/fingpt-qlora.git"
WORKDIR = Path("/kaggle/working")
REPO_DIR = WORKDIR / "fingpt-qlora"
ADAPTER_DATASET = "ericwang7717/fingpt-lora-adapter-v3"
ADAPTER_DIR = "outputs/lora_adapter"
OUTPUT_DIR = "results/eval_outputs/v3_completion_only"

os.chdir(WORKDIR)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL], check=True)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"Repo commit: {commit}")


In [ ]:
!python -m unittest discover tests
!python -m compileall -q src tests


In [ ]:
# Rebuild data splits so evaluation uses the current repository data contract.
!python -m src.data.download
!python -m src.data.preprocess
!python -m src.data.format_chat
!python -m src.data.merge_datasets
!python -m src.data.splits
!python -m src.data.manifest --splits-dir data/splits --output results/data_manifest_v3.json


In [ ]:
import os
os.makedirs(ADAPTER_DIR, exist_ok=True)
!kaggle datasets download -d {ADAPTER_DATASET} -p {ADAPTER_DIR} --unzip


In [ ]:
!python -m src.eval.run_evaluation \
  --test-data data/splits/test.json \
  --base-model unsloth/Qwen2.5-7B-Instruct-bnb-4bit \
  --adapter-path {ADAPTER_DIR} \
  --output-dir {OUTPUT_DIR} \
  --experiment-name v3_completion_only \
  --adapter-dataset {ADAPTER_DATASET}


In [ ]:
from pathlib import Path

for artifact in [
    f"{OUTPUT_DIR}/metrics.json",
    f"{OUTPUT_DIR}/predictions.jsonl",
    f"{OUTPUT_DIR}/comparison_table.md",
    "results/data_manifest_v3.json",
]:
    path = Path(artifact)
    print(f"{artifact}: {path.exists()}")
    assert path.exists(), artifact
